# 🌪️ NLP with Disaster Tweets — Advanced Pipeline
**Strategy:** twitter-roberta-base + soft-label ensemble + pseudo-labeling + threshold tuning → targeting 0.86+ F1

| Upgrade | Why it helps |
|---|---|
| `twitter-roberta-base` | Pre-trained on 58M tweets — understands slang, hashtags, mentions natively |
| Minimal preprocessing | RoBERTa tokenizer handles raw text better than heavily cleaned text |
| Soft probability ensemble | Average probabilities across folds instead of hard votes |
| Pseudo-labeling | High-confidence test predictions used as extra training data |
| Optimal threshold tuning | Find the best decision boundary on OOF probabilities |
| Location feature | Extra signal from the `location` column |


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import numpy as np

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print('Libraries loaded')


## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/sample_submission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
train.head()


## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train['target'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Not Disaster (0)', 'Disaster (1)'], rotation=0)

train['text_len'] = train['text'].str.len()
test['text_len']  = test['text'].str.len()
axes[1].hist(train[train['target']==0]['text_len'], bins=40, alpha=0.6, color='#2ecc71', label='Not Disaster')
axes[1].hist(train[train['target']==1]['text_len'], bins=40, alpha=0.6, color='#e74c3c', label='Disaster')
axes[1].set_title('Tweet Length by Class')
axes[1].legend()

train['word_count'] = train['text'].str.split().str.len()
axes[2].hist(train[train['target']==0]['word_count'], bins=30, alpha=0.6, color='#2ecc71', label='Not Disaster')
axes[2].hist(train[train['target']==1]['word_count'], bins=30, alpha=0.6, color='#e74c3c', label='Disaster')
axes[2].set_title('Word Count by Class')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Missing values:\n{train.isnull().sum()}')


## 4. Preprocessing
> **Key insight:** `twitter-roberta-base` was pre-trained on tweets, so it already understands 
> hashtags, mentions, and slang. We do **minimal** cleaning — only normalize URLs and fix 
> HTML entities. Over-cleaning (removing punctuation, numbers) actually hurts RoBERTa.


In [ ]:
def preprocess(text, keyword='', location=''):
    """
    Light preprocessing for twitter-roberta-base.
    Keep hashtags, mentions, punctuation — RoBERTa handles them natively.
    """
    text = str(text)
    # Normalize URLs to a token the model understands
    text = re.sub(r'http\S+|www\.\S+', 'http', text)
    # Fix HTML entities
    text = text.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    # Prepend keyword and location as context (separated by [SEP]-like marker)
    keyword  = str(keyword).replace('%20', ' ').strip()
    location = str(location).strip() if location and str(location) != 'nan' else ''
    
    prefix = ' '.join(filter(None, [keyword, location]))
    if prefix:
        text = prefix + ' | ' + text
    return text

# Fill NaNs first
train['keyword']  = train['keyword'].fillna('')
train['location'] = train['location'].fillna('')
test['keyword']   = test['keyword'].fillna('')
test['location']  = test['location'].fillna('')

train['input_text'] = train.apply(
    lambda r: preprocess(r['text'], r['keyword'], r['location']), axis=1)
test['input_text']  = test.apply(
    lambda r: preprocess(r['text'], r['keyword'], r['location']), axis=1)

print('Example:')
print('Original :', train['text'].iloc[0])
print('Input    :', train['input_text'].iloc[0])


## 5. Model Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────
# twitter-roberta-base: pre-trained on 58M tweets from Jan 2018–Dec 2021
# Understands tweet-specific language far better than general BERT/DistilBERT
MODEL_NAME  = 'cardiffnlp/twitter-roberta-base-2022-154m'
# Fallback if above is slow to download:
# MODEL_NAME = 'roberta-base'

MAX_LEN     = 128    # tweets are short; 128 is enough
BATCH_SIZE  = 16     # roberta-base is bigger than distilbert; reduce if OOM
EPOCHS      = 5      # more epochs with early stopping
N_FOLDS     = 5
LR          = 2e-5
WARMUP_FRAC = 0.1
SEED        = 42
PSEUDO_CONF = 0.97   # confidence threshold for pseudo-labeling

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Model  : {MODEL_NAME}')
print(f'Device : {DEVICE}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded')


## 6. Dataset & Training Utilities

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=MAX_LEN):
        self.encodings = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def predict_proba(model, loader):
    """Return soft probabilities (class 1) instead of hard labels."""
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            p = torch.softmax(outputs.logits, dim=-1)[:, 1]
            probs.extend(p.cpu().numpy())
    return np.array(probs)

print('✅ Dataset and training utilities defined')


## 7. 5-Fold Cross-Validation with Soft Ensemble
We collect **probability outputs** (not hard 0/1 predictions) from each fold and average them.  
This soft ensemble is more accurate than majority voting.


In [ ]:
X_texts  = train['input_text'].values
y_labels = train['target'].values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_probs  = np.zeros(len(train))   # out-of-fold probabilities
test_probs = np.zeros(len(test))    # averaged test probabilities

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_texts, y_labels)):
    print(f'\n══ Fold {fold+1}/{N_FOLDS} ══')

    tr_ds  = TweetDataset(X_texts[tr_idx],  y_labels[tr_idx])
    val_ds = TweetDataset(X_texts[val_idx], y_labels[val_idx])
    te_ds  = TweetDataset(test['input_text'].values)

    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2,              num_workers=2, pin_memory=True)
    te_loader  = DataLoader(te_ds,  batch_size=BATCH_SIZE*2,              num_workers=2, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    total_steps = len(tr_loader) * EPOCHS
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_FRAC),
        num_training_steps=total_steps
    )

    best_f1        = 0
    best_val_probs = None
    best_te_probs  = None
    patience       = 2
    no_improve     = 0

    for epoch in range(EPOCHS):
        loss      = train_epoch(model, tr_loader, optimizer, scheduler)
        val_probs = predict_proba(model, val_loader)
        val_preds = (val_probs >= 0.5).astype(int)
        f1        = f1_score(y_labels[val_idx], val_preds)
        print(f'  Epoch {epoch+1}/{EPOCHS} | loss={loss:.4f} | val F1={f1:.4f}')

        if f1 > best_f1:
            best_f1        = f1
            best_val_probs = val_probs
            best_te_probs  = predict_proba(model, te_loader)
            no_improve     = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    oof_probs[val_idx] = best_val_probs
    test_probs        += best_te_probs / N_FOLDS
    print(f'   Best fold F1: {best_f1:.4f}')

oof_preds_default = (oof_probs >= 0.5).astype(int)
print(f'\n OOF F1 (threshold=0.5): {f1_score(y_labels, oof_preds_default):.4f}')


## 8. Optimal Threshold Tuning
Instead of blindly using 0.5 as the decision boundary, we search for the threshold  
that maximises F1 on the OOF predictions.


In [ ]:
thresholds = np.arange(0.30, 0.70, 0.005)
f1_scores  = [f1_score(y_labels, (oof_probs >= t).astype(int)) for t in thresholds]

best_thresh = thresholds[np.argmax(f1_scores)]
best_oof_f1 = max(f1_scores)

# Plot
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_scores, color='#2980b9', linewidth=2)
plt.axvline(best_thresh, color='#e74c3c', linestyle='--', label=f'Best threshold = {best_thresh:.3f}')
plt.axhline(best_oof_f1, color='#27ae60', linestyle=':', label=f'Best OOF F1 = {best_oof_f1:.4f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('Threshold vs OOF F1 Score')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best threshold : {best_thresh:.3f}')
print(f'Best OOF F1    : {best_oof_f1:.4f}')
print()
print(classification_report(y_labels, (oof_probs >= best_thresh).astype(int),
                             target_names=['Not Disaster', 'Disaster']))


## 9. Pseudo-Labeling
High-confidence test predictions (probability > 0.97 or < 0.03) are added to the training  
set as extra labeled examples, then the model is retrained on the augmented dataset.  
This leverages unlabeled test data for free.


In [ ]:
# Select high-confidence pseudo labels
pseudo_mask   = (test_probs >= PSEUDO_CONF) | (test_probs <= (1 - PSEUDO_CONF))
pseudo_texts  = test['input_text'].values[pseudo_mask]
pseudo_labels = (test_probs[pseudo_mask] >= 0.5).astype(int)

print(f'Pseudo-labeled samples : {pseudo_mask.sum()} / {len(test)}')
print(f'  → Disaster     : {pseudo_labels.sum()}')
print(f'  → Not Disaster : {(pseudo_labels == 0).sum()}')

# Augmented training set
aug_texts  = np.concatenate([X_texts, pseudo_texts])
aug_labels = np.concatenate([y_labels, pseudo_labels])
print(f'\nAugmented train size: {len(aug_texts)} (was {len(X_texts)})')


### 9a. Retrain on Augmented Dataset

In [ ]:
# Final model trained on ALL train + pseudo-labeled test data
# No cross-validation here — we use the full data for maximum signal

aug_ds     = TweetDataset(aug_texts, aug_labels)
te_ds_full = TweetDataset(test['input_text'].values)

aug_loader = DataLoader(aug_ds,     batch_size=BATCH_SIZE,   shuffle=True,  num_workers=2, pin_memory=True)
te_loader2 = DataLoader(te_ds_full, batch_size=BATCH_SIZE*2,                num_workers=2, pin_memory=True)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(DEVICE)

total_steps_aug = len(aug_loader) * 4
optimizer_aug = AdamW(final_model.parameters(), lr=1e-5, weight_decay=0.01)  # lower LR for fine-tuning
scheduler_aug = get_linear_schedule_with_warmup(
    optimizer_aug,
    num_warmup_steps=int(total_steps_aug * WARMUP_FRAC),
    num_training_steps=total_steps_aug
)

print('Training final model on augmented data...')
for epoch in range(4):
    loss = train_epoch(final_model, aug_loader, optimizer_aug, scheduler_aug)
    print(f'  Epoch {epoch+1}/4 | loss={loss:.4f}')

# Final test predictions
final_test_probs = predict_proba(final_model, te_loader2)

# Blend: 70% CV ensemble + 30% pseudo-retrained model
blended_probs = 0.70 * test_probs + 0.30 * final_test_probs
print('\n Final blended predictions ready')


## 10. Generate Submission

In [ ]:
# Apply the optimal threshold found on OOF
final_preds = (blended_probs >= best_thresh).astype(int)

submission = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/sample_submission.csv')
submission['target'] = final_preds
submission.to_csv('submission.csv', index=False)

print(f'Submission saved ')
print(f'Predicted Disaster     : {final_preds.sum()}')
print(f'Predicted Not Disaster : {(final_preds == 0).sum()}')
submission.head(10)


## 11. Summary of Improvements

| Component | Old Notebook | This Notebook |
|---|---|---|
| **Model** | `distilbert-base-uncased` | `twitter-roberta-base` (tweet-native) |
| **Preprocessing** | Heavy cleaning (remove nums, punct) | Minimal (keep hashtags, mentions) |
| **Location** | Not used | Prepended as context |
| **Predictions** | Hard vote (0/1) per fold | Soft probabilities averaged |
| **Threshold** | Fixed 0.5 | Optimised on OOF |
| **Extra data** | None | Pseudo-labeled test samples |
| **Final model** | Fold ensemble only | Blend of ensemble + pseudo-retrained |
| **Early stopping** | No | Yes (patience=2) |

**Expected OOF F1:** ~0.84–0.87 depending on GPU and random seed.

###  If you want to push even higher:
- Try `roberta-large` (more parameters, slower)
- Add `DeBERTa-v3-base` to the ensemble for diversity
- Increase `PSEUDO_CONF` to 0.99 for stricter pseudo labels
- Use `focal loss` instead of cross-entropy for class imbalance
